In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16", "file_date")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.functions import col, lit

In [0]:
movies_df = (
    spark.read
        .table("movie_silver.movies")
        .filter(f"file_date = '{v_file_date}'")
).select("movie_id", "budget", "revenue", "year_release_date")


In [0]:
movies_genres_df = (
    spark.read
        .table("movie_silver.movie_genre")
        .filter(col("file_date") == v_file_date)
).select("genre_id", "movie_id")

In [0]:
genres_df = (
    spark.read
        .table("movie_silver.genres")
).select("genre_id", "genre_name")

In [0]:
results_group_movie_genre = (
    movies_df.filter(movies_df["year_release_date"] > 2000)
    .join(movies_genres_df,
        movies_df["movie_id"] == movies_genres_df["movie_id"], "inner")
    .join(genres_df,
            movies_genres_df["genre_id"] == genres_df["genre_id"])
).select("budget", "revenue", "year_release_date", "genre_name")

In [0]:
from pyspark.sql.functions import sum, avg, max, min, rank, dense_rank, desc, col, lit
from pyspark.sql.window import Window

In [0]:
budget_rank = Window.partitionBy("year_release_date").orderBy(desc("total_budget"))
revenue_rank = Window.partitionBy("year_release_date").orderBy(desc("total_revenue"))
budget_revenue_rank = (
    Window.partitionBy("year_release_date")
        .orderBy(
            desc("total_budget"), desc("total_revenue"))
)

results_group_movie_genre = (
    results_group_movie_genre
    .groupBy("year_release_date","genre_name")
        .agg(
            sum("budget").alias("total_budget"),
            sum("revenue").alias("total_revenue")  
        )
    .withColumn("budget_rank", rank().over(budget_rank))
    .withColumn("revenue_dense_rank", dense_rank().over(revenue_rank))
    .withColumn("budget_revenue_rank", rank().over(budget_revenue_rank))
    .withColumn("created_date", lit(v_file_date))
)
display(results_group_movie_genre)

In [0]:
#overwrite_partitioned_table("movie_silver", "results_group_movie_genre","created_date", v_file_date)

In [0]:
"""
(
    results_group_movie_genre
    .write
        .mode("append")
        .partitionBy("created_date")
        .saveAsTable(f"movie_gold.results_group_movie_genre")
)
"""

Utilizando Merge

In [0]:
merge_condition = 'trgt.year_release_date = src.year_release_date AND trgt.genre_name = src.genre_name AND trgt.created_date = src.created_date'
merge_delta_lake(input_df = results_group_movie_genre, schema_name = "movie_gold", table_name = "results_group_movie_genre", merge_condition = merge_condition, partition_column = "created_date")

In [0]:
%sql
SELECT created_date, COUNT(*)
FROM movie_gold.results_group_movie_genre
GROUP BY created_date
ORDER BY created_date